In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS medical_project.silver;

In [0]:
patients_df = spark.table("medical_project.bronze.patients")
display(patients_df.summary())

In [0]:
# Standardize Column Names
import re

def clean_column(col_name):
    col_name = col_name.strip().lower()
    col_name = re.sub(r"[^\w]", "_", col_name)   # replace spaces & special chars
    col_name = re.sub(r"_+", "_", col_name)      # remove multiple underscores
    col_name = col_name.strip("_")               # remove leading/trailing _
    return col_name

patients_df = patients_df.toDF(*[clean_column(c) for c in patients_df.columns])


# Remove Fivetran Metadata Columns
from pyspark.sql.functions import col

patients_df = patients_df.select([
    col(c) for c in patients_df.columns if not c.startswith("_")
])


# Convert Data Types
from pyspark.sql.functions import coalesce
from pyspark.sql.functions import try_to_date

patients_df = patients_df.withColumn("birthdate", 
                                      coalesce(try_to_date("birthdate", "yyyy-MM-dd"), 
                                              try_to_date("birthdate", "dd-MM-yyyy"))) \
                         .withColumn("death_date", 
                                      coalesce(try_to_date("death_date", "yyyy-MM-dd"), 
                                              try_to_date("death_date", "dd-MM-yyyy"))) \
                         .withColumn("zip_code", col("zip_code").cast("string")) \
                         .withColumn("lat", col("lat").cast("double")) \
                         .withColumn("lon", col("lon").cast("double"))


# Handle Null Values
patients_df = patients_df.fillna({
    "maiden": "NA",
    "suffix": "NA",
    "prefix": "NA"
})


# Remove Duplicates
patients_df = patients_df.dropDuplicates(["id"])


# Standardize Categorical Values
from pyspark.sql.functions import lower, trim

patients_df = patients_df.withColumn("gender", lower(trim(col("gender")))) \
                         .withColumn("race", lower(trim(col("race")))) \
                         .withColumn("ethnicity", lower(trim(col("ethnicity")))) \
                         .withColumn("marital_status", lower(trim(col("marital_status"))))


# Basic Validation (Remove Invalid Records)
patients_df = patients_df.filter(col("id").isNotNull())




In [0]:
# Write to Silver Layer
patients_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.silver.patients")